In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#!ls "/content/drive/MyDrive/Colab Notebooks/"

In [ ]:
#%run "/content/drive/MyDrive/Colab Notebooks/windows_logs.ipynb"
#%run "/content/drive/MyDrive/Colab Notebooks/windows_ai.ipynb"


In [ ]:
# #ai_train
# import pandas as pd
# import joblib

# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# from sklearn.metrics import precision_score, recall_score, f1_score
# FILE_PATH="/content/drive/MyDrive/Mini_Project/"

# # ===============================
# # LOAD BALANCED TRAIN DATA
# # ===============================

# df=pd.read_csv(FILE_PATH+"windows_labeled_logs_dataset.csv")

# print("Training logs:",len(df))

# # ===============================
# # FEATURES
# # ===============================

# features=[

# "event_id",
# "hour",
# "night_activity",
# "event_freq",
# "burst_count"

# ]

# X=df[features]

# y=df["label"]

# # ===============================
# # TRAIN TEST SPLIT
# # ===============================

# X_train,X_test,y_train,y_test=train_test_split(

# X,
# y,
# test_size=0.2,
# random_state=42,
# stratify=y

# )

# # ===============================
# # RANDOM FOREST MODEL
# # ===============================

# model=RandomForestClassifier(

# n_estimators=600,
# max_depth=18,
# min_samples_split=4,

# class_weight="balanced",

# n_jobs=-1,
# random_state=42

# )

# model.fit(X_train,y_train)

# # ===============================
# # EVALUATION
# # ===============================

# y_pred=model.predict(X_test)

# # Core metrics
# accuracy = accuracy_score(y_test, y_pred)
# precision = precision_score(y_test, y_pred, average='weighted')
# recall = recall_score(y_test, y_pred, average='weighted')
# f1 = f1_score(y_test, y_pred, average='weighted')

# print(f"\nAccuracy: {accuracy:.4f}")
# print(f"Precision: {precision:.4f}")
# print(f"Recall: {recall:.4f}")
# print(f"F1 Score: {f1:.4f}")
# print("\nAccuracy:",accuracy_score(y_test,y_pred))

# print("\nConfusion Matrix")
# print(confusion_matrix(y_test,y_pred))

# print("\nClassification Report")
# print(classification_report(y_test,y_pred))

# # ===============================
# # SAVE MODEL
# # ===============================

# package={

# "model":model,
# "features":features

# }

# joblib.dump(package,FILE_PATH+"windows_threat_model.pkl")

# print("\nModel saved successfully")

In [ ]:
#!ls "/content/drive/MyDrive/"

In [ ]:
# ===============================
# IMPORTS
# ===============================
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

# ===============================
# CONFIG
# ===============================
FILE_PATH = "/content/drive/MyDrive/Mini_Project/"
MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/" + "windows_threat_model.pkl"

TARGET_RECALL = 0.97   # relaxed for better precision
MIN_THRESHOLD = 0.05   # avoid extreme low threshold

# ===============================
# LOAD DATA
# ===============================
paths = [
    FILE_PATH + "windows_labeled_logs_dataset1.csv",
    FILE_PATH + "windows_labeled_logs_dataset2.csv",
    FILE_PATH + "windows_labeled_logs_dataset3.csv"
]

df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)

print("Total logs:", len(df))

# ===============================
# STANDARDIZE
# ===============================
df.columns = df.columns.str.strip().str.replace(" ", "_").str.lower()
df = df.dropna()

# ===============================
# FORCE BINARY LABEL
# ===============================
df["label"] = (df["label"] != 0).astype(int)

# ===============================
# FEATURE ENGINEERING (MATCH windows_ai.ipynb)
# ===============================
df["date_and_time"] = pd.to_datetime(
    df["date_and_time"],
    format="%d.%m.%Y. %H:%M:%S",
    errors="coerce"
)

df = df.sort_values("date_and_time")

df["hour"] = df["date_and_time"].dt.hour
df["night_activity"] = ((df["hour"] < 6) | (df["hour"] > 22)).astype(int)

df["event_freq"] = df.groupby("event_id").cumcount()
df["burst_count"] = df.groupby("event_id").cumcount().clip(upper=20)

# ===============================
# FEATURES (STRICT MATCH)
# ===============================
features = [
    "event_id",
    "hour",
    "night_activity",
    "event_freq",
    "burst_count"
]

X = df[features]
y = df["label"]

# ===============================
# SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ===============================
# CLASS IMBALANCE
# ===============================
pos = sum(y_train == 1)
neg = sum(y_train == 0)
scale_pos_weight = neg / pos

print("scale_pos_weight:", scale_pos_weight)

# ===============================
# MODEL
# ===============================
base_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=500,
    max_depth=7,
    learning_rate=0.07,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

base_model.fit(X_train, y_train)

# ===============================
# CALIBRATION (FIXED VERSION)
# ===============================
calibrator = CalibratedClassifierCV(base_model, method="sigmoid", cv=3)
calibrator.fit(X_train, y_train)

model = calibrator

# ===============================
# PROBABILITIES
# ===============================
y_probs = model.predict_proba(X_test)[:, 1]

# ===============================
# THRESHOLD OPTIMIZATION (IMPROVED)
# ===============================
best_threshold = 0.2
best_precision = 0

for t in np.arange(MIN_THRESHOLD, 0.5, 0.01):

    y_pred = (y_probs >= t).astype(int)

    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)

    # constraint: good recall
    if recall >= TARGET_RECALL:
        if precision > best_precision:
            best_precision = precision
            best_threshold = t

print("\nSelected Threshold:", best_threshold)

# ===============================
# FINAL EVALUATION
# ===============================
y_pred = (y_probs >= best_threshold).astype(int)

print("\n=== FINAL MODEL ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

# ===============================
# SAVE
# ===============================

package = {
    "model": model,
    "features": features,
    "threshold": best_threshold,
    "metrics": {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float( precision_score(y_test, y_pred)),
        "recall": float( recall_score(y_test, y_pred)),
        "f1": float(f1_score(y_test, y_pred)),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "classification_report": classification_report(y_test, y_pred)
    }
}

joblib.dump(package, MODEL_PATH)
# joblib.dump({
#     "model": model,
#     "features": features,
#     "threshold": best_threshold
# }, MODEL_PATH)

print("\nModel saved successfully!")

Total logs: 84065
scale_pos_weight: 0.4351073364346379


In [ ]:
# import pandas as pd
# import joblib

# FILE_PATH="/content/drive/MyDrive/Colab Notebooks/"

# # ==============================
# # LOAD MODEL
# # ==============================

# package = joblib.load(FILE_PATH+"windows_threat_detection_model.pkl")

# model = package["model"]
# features = package["features"]

# # ==============================
# # LOAD DATA
# # ==============================

# df = pd.read_csv(FILE_PATH+"windows_processed_logs.csv")

# if "label" in df.columns:
#     df = df.drop(columns=["label"])

# print("Logs loaded:",len(df))

# # ==============================
# # PREDICTION
# # ==============================

# X = df[features]

# df["risk_score"] = model.predict(X)
# print(df["risk_score"].value_counts())